In [115]:
import pandas as pd 
from pathlib import Path 
import numpy as np

In [116]:
Raw_path =Path(r"E:\New_portfolio_projects\E_Commerce_BI_Dataset")

In [117]:
print('Folder Exists',Raw_path.exists())

Folder Exists True


In [118]:
files = {"Customers":"olist_customers_dataset.csv",
        "geolocation":"olist_geolocation_dataset.csv",
        "order_items":"olist_order_items_dataset.csv",
        "order_payments":"olist_order_payments_dataset.csv",
        "order_reviews":"olist_order_reviews_dataset.csv",
        "orders_dataset":"olist_orders_dataset.csv",
         "products_dataset":"olist_products_dataset.csv",
         "sellers":"olist_sellers_dataset.csv",
         "product_category_name_translation":"product_category_name_translation.csv"
        }

dfs={}
for table_name , file in files.items():
    path = f"{Raw_path}\\{file}"
    dfs[table_name] =pd.read_csv(path)
    print(f"{table_name} loaded_with {dfs[table_name].shape[0]} rows and {dfs[table_name].shape[1]} columns")
    
    

Customers loaded_with 99441 rows and 5 columns
geolocation loaded_with 1000163 rows and 5 columns
order_items loaded_with 112650 rows and 7 columns
order_payments loaded_with 103886 rows and 5 columns
order_reviews loaded_with 99224 rows and 7 columns
orders_dataset loaded_with 99441 rows and 8 columns
products_dataset loaded_with 32951 rows and 9 columns
sellers loaded_with 3095 rows and 4 columns
product_category_name_translation loaded_with 71 rows and 2 columns


In [119]:
 for name , df in dfs.items():
     print(name , df.shape)


Customers (99441, 5)
geolocation (1000163, 5)
order_items (112650, 7)
order_payments (103886, 5)
order_reviews (99224, 7)
orders_dataset (99441, 8)
products_dataset (32951, 9)
sellers (3095, 4)
product_category_name_translation (71, 2)


In [120]:
def inspect (name,df):
      print(f"Name of datafram is  {name} with {df.shape[0]} rows and {df.shape[1]} columns")
      print("Data Types") 
      print(df.dtypes)
      print("sample of data")
      print(df.head(3))
      print("count of null values ")
      print(df.isnull().sum())
      print("count of duplicates")
      print(df.duplicated().sum())

In [121]:
for name , df in dfs.items():
    inspect(name,df)

Name of datafram is  Customers with 99441 rows and 5 columns
Data Types
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
sample of data
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
count of null values 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state   

In [122]:
# referential integrity check ( check the relationship between orders_items and orders  every order in order items must be in order df)
# expected null set if the relationship is true
order_and_order_items_rel_check =  set(dfs["order_items"]["order_id"])- set(dfs["orders_dataset"]["order_id"])
order_and_order_items_rel_check

set()

In [123]:
# referential integrity check ( check the relationship between orders and customer (if any customer make order did not found in customer df)
# expected null set if the relationship is true 
order_and_customer_rel_check = set(dfs["orders_dataset"]["customer_id"]) - set(dfs["Customers"]["customer_id"])
order_and_customer_rel_check

set()

In [124]:
product_order_item_rel_check = set(dfs["products_dataset"]["product_id"])  - set(dfs["order_items"]["product_id"])
product_order_item_rel_check                                 

set()

In [125]:
seller_order_item_rel_check = set(dfs["sellers"]["seller_id"])  - set(dfs["order_items"]["seller_id"])
seller_order_item_rel_check                                 

set()

In [126]:
order_and_order_payment_rel_check =  set(dfs["order_payments"]["order_id"])- set(dfs["orders_dataset"]["order_id"])
order_and_order_payment_rel_check

set()

In [127]:
# Check 2: multiple payment rows per order?
pay_per_order = dfs["order_payments"].groupby("order_id").size()
multi_pay     = (pay_per_order > 1).sum()
print(f"Orders with multiple payment rows: {multi_pay:,}")

Orders with multiple payment rows: 2,961


In [128]:
review_per_order = dfs["order_reviews"].groupby("order_id").size()
multi_review_per_order = (review_per_order >1).sum()
print(f"Orders with multiple reviews {multi_review_per_order }")

Orders with multiple reviews 547


In [129]:
# the customers df has two id the first one customer_id is unique for every order customer_unique_id is unique identifier for all customers 
print(f"totla rows in customers datafram {len(dfs["Customers"]):,}")
print(f"totla counts of customer_id datafram {(dfs["Customers"]["customer_id"]).count():,}")
print(f"totla rows in unique customer_id datafram {(dfs["Customers"]["customer_id"]).nunique():,}")
print(f"unique customer_unique_id values {(dfs["Customers"]["customer_unique_id"]).nunique():,}")



totla rows in customers datafram 99,441
totla counts of customer_id datafram 99,441
totla rows in unique customer_id datafram 99,441
unique customer_unique_id values 96,096


In [130]:
#actual repeate purches rate 
orders_per_person = (dfs["Customers"].merge(dfs["orders_dataset"], on=["customer_id"]).groupby("customer_unique_id")["order_id"].count())
loyal_customers_count = (orders_per_person >1).sum()
total = orders_per_person.shape[0]
print(f"loyal customers count",{loyal_customers_count})
print(f"Total{orders_per_person.shape[0]}")
print(f"loyal customers {loyal_customers_count} out of {total}")
print(f"percentage of loyal customers {loyal_customers_count /total*100}%")

loyal customers count {np.int64(2997)}
Total96096
loyal customers 2997 out of 96096
percentage of loyal customers 3.1187562437562435%


In [131]:
dfs["order_items"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [132]:
def clean_orders_datafram (df):
    df = df.copy()
#convert date columns from object  type to  date type 

    date_columns = [
        "order_purchase_timestamp",      
        "order_approved_at",                
        "order_delivered_carrier_date",     
        "order_delivered_customer_date", 
        "order_estimated_delivery_date"    
        
    ]
    for col in date_columns:
        df[col] = pd.to_datetime(df[col],errors="coerce",format='mixed')

    # BEFORE calculating delays, check if conversion worked
    print(f"Date column dtypes after conversion:")
    for col in date_columns:
        print(f"  {col}: {df[col].dtype}")

# add some insghitfull columns 
        df["delivery_delay_days"]= (df["order_estimated_delivery_date"]-df["order_delivered_customer_date"]).dt.days
        df["processing_days"] = (df["order_delivered_carrier_date"]-df["order_approved_at"]).dt.days
        df["is_late"] = df["delivery_delay_days"].apply(
            lambda x: True if pd.notnull(x) and x > 0
                 else False if pd.notnull(x) and x < 0 
                 else None )


    #clean order status 
    status_map = {
        "delivered":   "Delivered",
        "shipped":     "Shipped",
        "canceled":    "Cancelled",
        "unavailable": "Cancelled",
        "invoiced":    "Processing",
        "processing":  "Processing",
        "approved":    "Processing",
        "created":     "Processing"
    }
    
    df["order_status_clean"] = df["order_status"].map(status_map).fillna("other")
    # Extract the date from timestamp date column 
    df["purchase_date"] = df["order_purchase_timestamp"].dt.date
    print(f"cleaned orders {df.shape} ")
    return df
    

In [133]:
# check 

In [134]:
orders = dfs["orders_dataset"]

In [135]:
orders_cleaned = clean_orders_datafram(orders)
orders_cleaned 

Date column dtypes after conversion:
  order_purchase_timestamp: datetime64[ns]
  order_approved_at: datetime64[ns]
  order_delivered_carrier_date: datetime64[ns]
  order_delivered_customer_date: datetime64[ns]
  order_estimated_delivery_date: datetime64[ns]
cleaned orders (99441, 13) 


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,processing_days,is_late,order_status_clean,purchase_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7.0,2.0,True,Delivered,2017-10-02
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,5.0,0.0,True,Delivered,2018-07-24
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,17.0,0.0,True,Delivered,2018-08-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,12.0,3.0,True,Delivered,2017-11-18
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,9.0,0.0,True,Delivered,2018-02-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,10.0,1.0,True,Delivered,2017-03-09
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,1.0,1.0,True,Delivered,2018-02-06
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,5.0,1.0,True,Delivered,2017-08-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,20.0,3.0,True,Delivered,2018-01-08


In [136]:
orders_cleaned.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
delivery_delay_days                     float64
processing_days                         float64
is_late                                  object
order_status_clean                       object
purchase_date                            object
dtype: object

In [137]:
xx = orders_cleaned[orders_cleaned['is_late'] == False]
xx

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,processing_days,is_late,order_status_clean,purchase_date
20,203096f03d82e0dffbc41ebc2e2bcfb7,d2b091571da224a1b36412c18bc3bbfe,delivered,2017-09-18 14:31:30,2017-09-19 04:04:09,2017-10-06 17:50:03,2017-10-09 22:23:46,2017-09-28,-12.0,17.0,False,Delivered,2017-09-18
25,fbf9ac61453ac646ce8ad9783d7d0af6,3a874b4d4c4b6543206ff5d89287f0c3,delivered,2018-02-20 23:46:53,2018-02-22 02:30:46,2018-02-26 22:25:22,2018-03-21 22:03:54,2018-03-12,-10.0,4.0,False,Delivered,2018-02-20
35,8563039e855156e48fccee4d611a3196,5f16605299d698660e0606f7eae2d2f9,delivered,2018-02-17 15:59:46,2018-02-17 16:15:34,2018-02-20 23:03:56,2018-03-20 00:59:25,2018-03-20,-1.0,3.0,False,Delivered,2018-02-17
41,6ea2f835b4556291ffdc53fa0b3b95e8,c7340080e394356141681bd4c9b8fe31,delivered,2017-11-24 21:27:48,2017-11-25 00:21:09,2017-12-13 21:14:05,2017-12-28 18:59:23,2017-12-21,-8.0,18.0,False,Delivered,2017-11-24
57,66e4624ae69e7dc89bd50222b59f581f,684fa6da5134b9e4dab731e00011712d,delivered,2018-03-09 14:50:15,2018-03-09 15:40:39,2018-03-15 00:31:19,2018-04-03 13:28:46,2018-04-02,-2.0,5.0,False,Delivered,2018-03-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99396,b3112ca67f3afd4e20cf2ee91fc4f804,6f83c71b6c044fb156d697d4130fe9b5,delivered,2018-08-02 22:46:54,2018-08-02 23:04:06,2018-08-15 17:42:00,2018-08-21 00:03:26,2018-08-16,-6.0,12.0,False,Delivered,2018-08-02
99397,0fa1fab1d7c1211c824596ed5e111e3c,7f3bd6c94d2daf7b6462d1a894a775b4,delivered,2018-03-13 21:48:57,2018-03-13 22:40:28,2018-03-14 19:27:23,2018-04-05 19:59:49,2018-04-02,-4.0,0.0,False,Delivered,2018-03-13
99423,38e9133ce29f6bbe35aed9c3863dce01,ad312389a098ceff46ce92c4595c06d0,delivered,2017-10-12 20:54:11,2017-10-14 03:28:24,2017-10-17 17:04:42,2017-11-21 17:06:59,2017-10-31,-22.0,3.0,False,Delivered,2017-10-12
99425,d692ef54145c9cb3322ec2e5508aa3f4,82ddfcf9438b0cd1117b55ac33184df8,delivered,2018-03-21 19:47:18,2018-03-21 20:05:26,2018-03-22 21:11:58,2018-04-11 00:48:31,2018-04-09,-3.0,1.0,False,Delivered,2018-03-21


In [138]:
order_items = dfs["order_items"]
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [139]:
def clean_order_items_df(df):
    df = df.copy()
    
    # change columns data type to the correct data type  
    df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"],  errors="coerce").fillna(0)
    df["freight_value"] = pd.to_numeric(df["freight_value"],  errors="coerce").fillna(0)

    #add columns total_price_per_product (price + freight_value)
    df["total_item_value"] = df["price"] + df["freight_value"]
    df["freight_ratio"] = df["freight_value"]/ df["total_item_value"].replace(0,np.nan)
    print(f"cleaned order_items done: {df.shape}")
    return df

In [140]:
order_items = dfs["order_items"]
order_items
print(order_items.dtypes)

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object


In [141]:
#check 

In [142]:
clean_order_items_df(order_items)

cleaned order_items done: (112650, 9)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_item_value,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,0.082400
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,0.496121
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,0.083196
...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40,0.126412
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53,0.094508
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85,0.145058
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71,0.134755


In [143]:
dfs["geolocation"].shape

(1000163, 5)

In [144]:
dfs["geolocation"]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [145]:
def Clean_Geolocation_df (df):
    df = df.copy()
    geolocation_Aggregated = df.groupby("geolocation_zip_code_prefix")[["geolocation_lat","geolocation_lng"]].first().reset_index().rename(columns={"geolocation_zip_code_prefix":"zip_code",
                                                                                                                                       'geolocation_lat': 'latitude',
                                                                                                                                       'geolocation_lng':'longitude'})
    print(f"Collapse Geolocation from {df.shape[0]} GPS Coordinates for every Zip code to {geolocation_Aggregated.shape[0]} for the first value")
    return geolocation_Aggregated 

In [146]:
geo = dfs["geolocation"]


In [147]:
Clean_Geolocation_df(geo)

Collapse Geolocation from 1000163 GPS Coordinates for every Zip code to 19015 for the first value


,zip_code,latitude,longitude
0,1001,-23.549292,-46.633559
1,1002,-23.548318,-46.635421
2,1003,-23.549032,-46.635313
3,1004,-23.550116,-46.635122
4,1005,-23.549819,-46.635606
...,...,...,...
19010,99960,-27.953797,-52.029641
19011,99965,-28.173892,-52.038447
19012,99970,-28.345143,-51.876926
19013,99980,-28.389218,-51.846012


In [148]:
def clean_customers_df(Customers_df,geo_df):
    df = Customers_df.copy()
    df = df.rename(columns={
        "customer_zip_code_prefix": "zip_code",
        "customer_city": "city",
        "customer_state": "state"
    })

        # Map state codes to Brazil state 
    state_map = {
    "SP": "São Paulo",
    "RJ": "Rio de Janeiro",
    "MG": "Minas Gerais",
    "ES": "Espírito Santo",
    "PR": "Paraná",
    "SC": "Santa Catarina",
    "RS": "Rio Grande do Sul",
    "BA": "Bahia",
    "PE": "Pernambuco",
    "CE": "Ceará",
    "MA": "Maranhão",
    "PB": "Paraíba",
    "RN": "Rio Grande do Norte",
    "AL": "Alagoas",
    "SE": "Sergipe",
    "PI": "Piauí",
    "AM": "Amazonas",
    "PA": "Pará",
    "RO": "Rondônia",
    "AC": "Acre",
    "AP": "Amapá",
    "RR": "Roraima",
    "TO": "Tocantins",
    "MT": "Mato Grosso",
    "GO": "Goiás",
    "MS": "Mato Grosso do Sul",
    "DF": "Distrito Federal"}

    df["state_name"] = df["state"].map(state_map).fillna("Unknown")




    #add region column
    region_map = {
    "SP":"Southeast","RJ":"Southeast","MG":"Southeast","ES":"Southeast",
    "PR":"South",    "SC":"South",    "RS":"South",
    "BA":"Northeast","PE":"Northeast","CE":"Northeast","MA":"Northeast",
    "PB":"Northeast","RN":"Northeast","AL":"Northeast","SE":"Northeast","PI":"Northeast",
    "AM":"North",    "PA":"North",    "RO":"North",    "AC":"North",
    "AP":"North",    "RR":"North",    "TO":"North",
    "MT":"Midwest",  "GO":"Midwest",  "MS":"Midwest",  "DF":"Midwest"}

  
    df["region"]     = df["state"].map(region_map).fillna("Other")

    #merge coordinates from geolocation df to customer df 
    
    geo = Clean_Geolocation_df(geo_df)
    df = df.merge(geo ,on="zip_code", how="left")
    print(F"clean customers done with {df.shape[0]} rows and {df.shape[0]} columns")
    return df

    

In [149]:
customers = dfs["Customers"]
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [150]:
clean_customers = clean_customers_df(customers,geo)
clean_customers

Collapse Geolocation from 1000163 GPS Coordinates for every Zip code to 19015 for the first value
clean customers done with 99441 rows and 99441 columns


,customer_id,customer_unique_id,zip_code,city,state,state_name,region,latitude,longitude
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,São Paulo,Southeast,-20.509897,-47.397866
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,São Paulo,Southeast,-23.726853,-46.545746
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,São Paulo,Southeast,-23.527788,-46.660310
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,São Paulo,Southeast,-23.496930,-46.185352
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,São Paulo,Southeast,-22.987222,-47.151073
...,...,...,...,...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP,São Paulo,Southeast,-23.587901,-46.501830
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP,São Paulo,Southeast,-23.612294,-46.765787
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE,Ceará,Northeast,-3.744128,-38.510859
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS,Rio Grande do Sul,South,-29.956391,-51.167614


In [151]:
def Clean_product_df(df,translation_df):
    df = df.copy()
    df["product_category_name"] = df["product_category_name"].fillna("Unknown")
    df["product_name_lenght"] = df["product_name_lenght"].fillna("Unknown")
    translation = translation_df.copy()
    df = df.merge(translation,on ="product_category_name",how="left")

        # Group 70+ categories into 8 broad groups for cleaner Power BI visuals
    def categorize(cat):
        cat = str(cat).lower()
        if any(w in cat for w in ["fashion","sport","luggage","leisure"]):
            return "Fashion & Sports"
        elif any(w in cat for w in ["electronics","computer","phone","tablet","audio","auto","telephony","watches_gifts","consoles_game","cine_photo"]):
            return "Electronics"
        elif any(w in cat for w in ["furniture","home","garden","kitchen","bed","housewares","construction_tools_lights","construction_tools_safety","construction_tools_construction","signaling_and_security"]):
            return "Home & Garden"
        elif any(w in cat for w in ["health","beauty","perfume","cosmetic"]):
            return "Health & Beauty"
        elif any(w in cat for w in ["book","cd","dvd","music","art"]):
            return "Books & Media"
        elif any(w in cat for w in ["toy","baby","kids"]):
            return "Toys & Baby"
        elif any(w in cat for w in ["stationery"]):
            return "stationery"
        elif any(w in cat for w in ["pet_shop"]):
            return "pet_shop"  
        elif any(w in cat for w in ["food","drink","grocery"]):
            return "Food & Drinks"
        else:
            return "Other"
    df["category_group"] = df["product_category_name_english"].apply(categorize)


    return df 

In [152]:
product = dfs["products_dataset"]
product

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [153]:
x = Clean_product_df(dfs["products_dataset"],dfs["product_category_name_translation"])
x[x["category_group"]=="Other"]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category_group
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0,cool_stuff,Other
44,7a8dac4aaa16bc642e4df33adcf03303,cool_stuff,38.0,1701.0,2.0,11750.0,18.0,40.0,79.0,cool_stuff,Other
46,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,38.0,1176.0,3.0,15350.0,47.0,40.0,47.0,cool_stuff,Other
51,b5cfb1d3c5e435a7a52227e08f220ee7,cool_stuff,55.0,410.0,2.0,1300.0,36.0,16.0,24.0,cool_stuff,Other
57,fdeb34a9f03fea7c3937dd62d1d0287e,cool_stuff,58.0,1252.0,1.0,1100.0,30.0,20.0,23.0,cool_stuff,Other
...,...,...,...,...,...,...,...,...,...,...,...
32772,fa51e914046aab32764c41356b9d4ea4,Unknown,Unknown,NaN,NaN,1300.0,45.0,16.0,45.0,NaN,Other
32852,c4ceee876c82b8328e9c293fa0e1989b,Unknown,Unknown,NaN,NaN,700.0,28.0,3.0,43.0,NaN,Other
32892,cd96152f38323aa8177e3a72d6558518,cool_stuff,37.0,510.0,1.0,1367.0,28.0,25.0,33.0,cool_stuff,Other
32897,ae7937db16790e94d48df1991877b508,cool_stuff,58.0,860.0,3.0,400.0,16.0,10.0,12.0,cool_stuff,Other


In [154]:
dfs["order_payments"].dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [155]:
dfs["order_payments"]["payment_installments"].unique()

array([ 8,  1,  2,  3,  6,  5,  4, 10,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23])

In [156]:
def clean_payments_df(df):
    df = df.copy()
    # Just clean types — no collapsing
    df['payment_value']        = pd.to_numeric(df['payment_value'],        errors='coerce').fillna(0)
    df['payment_installments'] = pd.to_numeric(df['payment_installments'], errors='coerce').fillna(0).astype(int)
    df['payment_type']         = df['payment_type'].str.strip().str.lower()

    def bucket(n):
        if n == 1: return 'cash'
        elif n <= 3: return '2-3'
        elif n <= 6: return '3-6'
        elif n <= 9: return '6-9'
        elif n <= 12: return '9-12'
        else : return '+12'
    df["installments_buckets"] = df['payment_installments'].apply(bucket)
    print(f"Clean Payments done with shape {df.shape}")    
    return df

In [157]:
payment = dfs["order_payments"].copy()

In [158]:
clean_payment =  clean_payments_df(payment)
clean_payment

Clean Payments done with shape (103886, 6)


,order_id,payment_sequential,payment_type,payment_installments,payment_value,installments_buckets
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,6-9
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,cash
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,cash
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,6-9
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2-3
...,...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31,cash
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80,2-3
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77,cash
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54,3-6


In [159]:
dfs["order_reviews"].dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [160]:
def clean_Reviews_df(df):
    df = df.copy()
    #convert the date time columns from date object to date 
    df["review_creation_date"] = pd.to_datetime(df["review_creation_date"])
    df["review_answer_timestamp"] =pd.to_datetime(df["review_answer_timestamp"])

    #some orders have multiple  reviews so i will keep the lasted review per order 
    df = df.sort_values("review_creation_date",ascending=False).drop_duplicates(subset="order_id",keep="first")
    #Add sentiment label
    score_map = {1:"Very Negative", 2:"Negative", 3:"Neutral",
                 4:"Positive",      5:"Very Positive"}
    df["review_sentiment"] = df["review_score"].map(score_map)

    # Did the customer write a comment?
    df["has_comment"] = (
        df["review_comment_message"].notna()
        & (df["review_comment_message"].str.strip() != "")
    )

    print(f"Clean Reviews done with shape {df.shape}")

    return df
    

In [161]:
df = dfs["order_reviews"].copy()
x = clean_Reviews_df(df)
x

Clean Reviews done with shape (98673, 9)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment,has_comment
10764,0574e3efaf2a10ee2d0c45b1ed544c84,bd2c2c3a4d59e68fb14a526745572883,4,NaN,NaN,2018-08-31,2018-09-01 12:27:54,Positive,False
55456,f542d0ff698f5272fbd13deca2349582,529a65336debb1c3e3327d7d67dc733d,5,Excelente,NaN,2018-08-31,2018-09-01 20:50:38,Very Positive,False
96338,02d00568beb770ce3939f4fca60556a1,b0e9288a209f5ec50391c140dba4c91f,5,NaN,NaN,2018-08-31,2018-09-05 22:33:44,Very Positive,False
47024,c4836c2b7d45087213b6ad924f90f679,d02b32c3bcfb76481817b2222b990e84,5,NaN,NaN,2018-08-31,2018-08-31 23:24:47,Very Positive,False
53698,0a85120e444360801174a4a5f641bb3e,8506571faf231af2bd4e43d1ba47dce8,5,NaN,NaN,2018-08-31,2018-08-31 19:07:10,Very Positive,False
...,...,...,...,...,...,...,...,...,...
60439,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,NaN,NaN,2016-10-15,2016-10-16 03:20:17,Positive,False
41042,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,NaN,Só chegou uma parte do pedido ate agora..,2016-10-15,2016-10-17 21:02:49,Positive,True
5503,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,NaN,"PRODUTO NÃO CHEGOU,E JÁ PASSOU O PRAZO DE ENTREGA",2016-10-09,2016-10-11 14:31:29,Very Negative,True
37547,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28,Very Negative,True


In [162]:
# transform  restructre the tables and built the dimensions and fact tables 

In [163]:
def build_dim_date(start="2016-01-01", end="2018-12-31"): # because the dataset  for The dataset has information of 100k orders from 2016 to 2018 
    dates = pd.date_range(start, end, freq="D")
    dim = pd.DataFrame({
        "date_key":     dates.strftime("%Y%m%d").astype(int),
        "full_date":    dates,
        "year":         dates.year,
        "quarter":      dates.quarter,
        "quarter_label": "Q" + dates.quarter.astype(str),
        "month":        dates.month,
        "month_name":   dates.strftime("%B"),
        "month_short":  dates.strftime("%b"),
        "week":         dates.isocalendar().week.astype(int),
        "day_of_week":  dates.strftime("%A"),
        "is_weekend":   dates.weekday >= 5,
        "is_month_end": dates.is_month_end,
    })
    print(f"dim_date: {dim.shape[0]:,} rows")
    return dim


In [164]:
def build_Dim_Customer(customers_df,geo_def):
    dim = clean_customers_df(customers_df,geo_def)
    keep = ["customer_id","customer_unique_id","zip_code","city","state","state_name","region","latitude","longitude"]
    dim = dim[[c for c in keep if c in dim.columns]].copy()
    dim = dim.reset_index(drop = True)
    dim.insert(0,"Customer_Sk", dim.index+1)
    print(f"dim_customer: {dim.shape[0]:,} rows | cols: {list(dim.columns)}")
    return dim

In [165]:
def build_Dim_Product(prouct_df, translation_df):
    dim = Clean_product_df(prouct_df, translation_df)
    keep = ["product_id","product_category_name","category_english","category_group"]
    dim = dim[[c for c in keep if c in dim.columns]].copy()
    dim = dim.reset_index(drop=True)
    dim.insert(0,"Product_sk",dim.index + 1)
    print(f"dim product with shape {dim.shape} rows | cols: {list(dim.columns)}")
    return dim
    

In [166]:
def build_dim_payments(payment_df):
    dim = clean_payments_df(payment_df)
    dim = dim.reset_index(drop = True)
    dim.insert(0,"Payments_sk",dim.index + 1)
    print(f"dim payments with shaape {dim.shape}")
    return dim 

In [167]:
def build_dim_seller(sellers_df, geo_df):
    df = sellers_df.copy()
    df = df.rename(columns={"seller_zip_code_prefix":"zip_code","seller_city":"city","seller_state":"state"})
    geo_agg = Clean_Geolocation_df(geo_df)
    df = df.merge(geo_agg, on="zip_code", how="left")
    df = df.reset_index(drop=True)
    df.insert(0, "seller_SK", df.index + 1)
    print(f"dim_seller: {df.shape[0]:,} rows")
    return df

In [168]:
def build_fact_orders(order_df, items_order_df, reviews_df, dim_customer, dim_product, dim_payment, dim_seller, dim_date):

    
    # Start with order items — this sets system grain (1 row per item)
    fact = clean_order_items_df(items_order_df)
    print(f"After clean_order_items: {fact.shape}, columns: {list(fact.columns)}")
    
    # Merge with orders
    orders_cleaned = clean_orders_datafram(order_df)
    print(f"After clean_orders: orders_cleaned.shape={orders_cleaned.shape}")
    fact = fact.merge(orders_cleaned, on="order_id", how='left')
    print(f"After merge orders: {fact.shape}")
    
    # Merge with reviews
    reviews_cleaned = clean_Reviews_df(reviews_df)
    print(f"After clean_reviews: {reviews_cleaned.shape}")
    fact = fact.merge(reviews_cleaned[["order_id", "review_score", "review_sentiment", "has_comment"]], on="order_id", how="left")
    print(f"After merge reviews: {fact.shape}")

    # Merge with dimensions
    print(f"\nDIM_PAYMENT CHECK:")
    print(f"  Shape: {dim_payment.shape}")
    print(f"  Columns: {dim_payment.columns.tolist()}")
    
    fact = fact.merge(dim_payment[['Payments_sk', 'order_id']], on="order_id", how="left")
    fact = fact.merge(dim_customer[["customer_id", "Customer_Sk"]], on="customer_id", how="left")
    fact = fact.merge(dim_product[["product_id", "Product_sk"]], on="product_id", how="left")
    fact = fact.merge(dim_seller[["seller_id", "seller_SK"]], on="seller_id", how="left")

    # Create date_key
    fact["purchase_date"] = pd.to_datetime(fact["purchase_date"])
    fact["date_key"] = fact["purchase_date"].dt.strftime("%Y%m%d").astype(int)

   
    fact.insert(0, "fact_SK", range(1, len(fact) + 1))

    print(f"\nfact_orders built: {fact.shape}")
    return fact  # ← FIXED: was "return f"

In [169]:
# etl/load.py
from sqlalchemy import create_engine, text
import pandas as pd
 
SERVER   = "KHADIJAA"
DATABASE = "EcommerceDWH"
 
# The connection string tells SQLAlchemy everything it needs:
# what database type, what server, what database, how to authenticate
CONN_STR = (
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
 
def get_engine():
    return create_engine(CONN_STR, fast_executemany=True)
 
 
def create_database():
    """Create the database if it does not already exist"""
    import pyodbc
    
    # Connect to master database to create new database
    conn_string = f"Driver={{ODBC Driver 17 for SQL Server}};Server={SERVER};Trusted_Connection=yes;"
    conn = pyodbc.connect(conn_string)
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Create database if it doesn't exist
    cursor.execute(f"IF NOT EXISTS (SELECT * FROM sys.databases WHERE name='{DATABASE}') CREATE DATABASE {DATABASE}")
    
    conn.close()
    print(f"Database '{DATABASE}' is ready.")
 
 
def load_table(df, table_name, engine):
    """Write a DataFrame to a SQL Server table"""
    df.to_sql(
        name      = table_name,
        con       = engine,
        if_exists = "replace",   # drop and recreate if table exists
        index     = False,       # do not write DataFrame row numbers as a column
        chunksize = 2000         # send 2000 rows per INSERT batch
    )
    print(f"  Loaded [{table_name}]: {len(df):,} rows")
 
 
def load_all(dim_customer, dim_product,dim_payment, dim_seller,  dim_date, fact_orders):
    create_database()
    engine = get_engine()
 
    print("Loading dimensions...")
    load_table(dim_customer, "dim_customer", engine)
    load_table(dim_product,  "dim_product",  engine)
    load_table(dim_seller,   "dim_seller",   engine)
    load_table(dim_payment,  "dim_payment",  engine)
    load_table(dim_date,     "dim_date",     engine)
 
    print("Loading fact table...")
    load_table(fact_orders,  "fact_orders",  engine)
 
    print("All tables loaded successfully.")


In [170]:
# etl/pipeline.py
import sys, time, pandas as pd
from pathlib import Path
 
sys.path.insert(0, ".")  # tells Python to look in the current folder for imports

RAW = Path(r"E:\New_portfolio_projects\E_Commerce_BI_Dataset")
 
def run():
    t0 = time.time()
    print("=" * 50)
    print("  E-COMMERCE BI PIPELINE — STARTED")
    print("=" * 50)

    # ── EXTRACT ────────────────────────────────────
    print("\n[1/3] Loading source files...")
    orders      = pd.read_csv(RAW / "olist_orders_dataset.csv")
    items       = pd.read_csv(RAW / "olist_order_items_dataset.csv")
    customers   = pd.read_csv(RAW / "olist_customers_dataset.csv")
    products    = pd.read_csv(RAW / "olist_products_dataset.csv")
    sellers     = pd.read_csv(RAW / "olist_sellers_dataset.csv")
    payments    = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
    reviews     = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", low_memory=False)
    geo         = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
    translation = pd.read_csv(RAW / "product_category_name_translation.csv")
    print("  All 9 files loaded.")

    # ── TRANSFORM ──────────────────────────────────
    print("\n[2/3] Transforming...")
    dim_customer = build_Dim_Customer(customers, geo)
    dim_product  = build_Dim_Product(products, translation)
    dim_seller   = build_dim_seller(sellers, geo)
    dim_payment  = build_dim_payments(payments)
    

    dim_date     = build_dim_date()
    fact         = build_fact_orders(
        orders, items, reviews,
        dim_customer, dim_product, dim_payment, dim_seller, dim_date
    )

    # ── LOAD
    print("\n[3/3] Loading to SQL Server...")
    load_all(dim_customer, dim_product, dim_payment, dim_seller,  dim_date, fact)

    elapsed = time.time() - t0
    print(f"\nPIPELINE COMPLETE in {elapsed:.1f} seconds")
    print("=" * 50)

if __name__ == "__main__":
    run()

  E-COMMERCE BI PIPELINE — STARTED

[1/3] Loading source files...
  All 9 files loaded.

[2/3] Transforming...
Collapse Geolocation from 1000163 GPS Coordinates for every Zip code to 19015 for the first value
clean customers done with 99441 rows and 99441 columns
dim_customer: 99,441 rows | cols: ['Customer_Sk', 'customer_id', 'customer_unique_id', 'zip_code', 'city', 'state', 'state_name', 'region', 'latitude', 'longitude']
dim product with shape (32951, 4) rows | cols: ['Product_sk', 'product_id', 'product_category_name', 'category_group']
Collapse Geolocation from 1000163 GPS Coordinates for every Zip code to 19015 for the first value
dim_seller: 3,095 rows
Clean Payments done with shape (103886, 6)
dim payments with shaape (103886, 7)
dim_date: 1,096 rows
cleaned order_items done: (112650, 9)
After clean_order_items: (112650, 9), columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'total_item_value', 'freight_ratio']
Da